# Interactive Dashboard
This dashboard explores the different weather conditions from data provided by the Dublin, San Jose, and Toronto Stations 

In [1]:
import pandas as pd
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
#!pip install --upgrade nbformat
import plotly.io as pio
pio.renderers.default="notebook"

In [6]:
weather = pd.read_csv("../data/weather_14-25_v2.csv", parse_dates=['DATE'])
weather['SEASON'] = pd.Categorical(weather.SEASON, categories=['Spring', 'Summer', 'Fall', 'Winter'], ordered=True)
weather['STATION'] = pd.Categorical(weather.STATION, categories=['Dublin', 'Toronto', 'San Jose'])
weather['YEAR'] = weather['DATE'].dt.year
weather['MONTH']=weather['DATE'].dt.month
weather.head()

,STATION,DATE,AWND,PRCP,SNOW,TAVG,TMAX,TMIN,SEASON,YEAR,MONTH
0,Dublin,2014-01-01,NaN,3.8,NaN,6.3,10.5,2.7,Winter,2014,1
1,Dublin,2014-01-02,NaN,3.7,NaN,6.8,10.3,4.7,Winter,2014,1
2,Dublin,2014-01-03,NaN,3.1,NaN,6.8,8.3,3.6,Winter,2014,1
3,Dublin,2014-01-04,NaN,0.0,NaN,3.9,7.2,1.2,Winter,2014,1
4,Dublin,2014-01-05,NaN,4.4,NaN,6.9,12.8,1.2,Winter,2014,1


In [11]:
# Groupings for animation (yearly)
seasonal_avg = (weather.groupby(['YEAR', 'SEASON', 'STATION'], as_index=False)
    ['TMAX'].mean())

season_order = ['Winter', 'Spring', 'Summer', 'Fall']
stations = ['Dublin', 'Toronto', 'San Jose']
years = sorted(seasonal_avg['YEAR'].unique())

In [28]:
#Maybe we can do both a dropdown for station and a year slider ? 

dash = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Seasonal Max Temperature Averages', 'title2',
                    'title3', 'title4')
)

# Creating the bargraphs (animation)
for station in stations:
    data = seasonal_avg[
        (seasonal_avg['YEAR'] == years[0]) &
        (seasonal_avg['STATION'] == station)
    ].set_index('SEASON').reindex(season_order)

    dash.add_trace(go.Bar( # Adds 3 traces, bars for each station
        x=season_order,
        y=data['TMAX'],
        name=station),
        row=1, col=1
    )

dash.frames = [
    go.Frame(name=str(year),
        data=[go.Bar(
                x=season_order,
                y=[seasonal_avg.loc[(seasonal_avg['YEAR'] == year) &
                (seasonal_avg['STATION'] == station) &
                (seasonal_avg['SEASON'] == season),'TMAX'].mean()
                for season in season_order])
            for station in stations],traces=[0, 1, 2]) # Animation will only be for plot 1 (trace 1-3)
    for year in years]

dash.update_layout(
    title='Dashboard - Title',
    height=800,
    width=1100,
    margin=dict(t=140, l=100, r=50, b=80),

    updatemenus=[{
        'type': 'buttons',
        'direction': 'left',
        'x': 0.1,
        'y': 1.1,
        'xanchor': 'left',
        'yanchor': 'top',
        'buttons': [
            {'label': 'Play', 'method': 'animate',
             'args': [None]},
            {'label': 'Pause', 'method': 'animate',
             'args': [[None], {
                 'mode': 'immediate',
                 'frame': {'duration': 0},
                 'transition': {'duration': 0}
             }]}
        ]
    }],
    sliders=[{ # You can remove this if you don't want the slider
        'steps': [
            {'method': 'animate', 'label': str(year),
             'args': [[str(year)]]}
            for year in years
        ]
    }]
)

dash.update_xaxes(title_text="Season", row=1, col=1)
dash.update_yaxes(title_text="Max Average Temperature (C)", range=[-3,30], row=1, col=1)
dash.show()